In [1]:
from dotenv import load_dotenv
from src.db.chroma_db import ChromaDb
from src.models.openai_provider import OpenAILLMProvider, OpenAIEmbeddingProvider
from src.prompts import timeframe_detection_system, timeframe_detection_user
from src.models.schemas import TimeframeDetection, InputLanguage
from src.services.nkod_data_processor import NkodDataProcessor
from src.db.graph_db import GraphDb
from src.db.sq_lite import SqLite
from datetime import date
from src.services.language_detector import LanguageDetector
from src.services.nkod_query_matcher import NkodQueryMatcher
from src.services.timeframe_detector import TimeframeDetector
from src.services.nkod_query_matcher_evaluator import NkodQueryMatcherEvaluator
from src.models.google_provider import GeminiEmbeddingProvider
from src.services.nkod_query_matcher_reranker import NkodQueryMatcherReranker


load_dotenv()

/home/lamossta/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

## Downloading and creating SPARQL endpoint for the NKOD metadata

In [2]:
nkod_data_processor = NkodDataProcessor("nkod")
graph_db = GraphDb(nkod_data_processor.catalog_name)
sq_lite = SqLite(nkod_data_processor.metadata_sql_path)

#nkod_data_processor.download_catalog_metadata()
#nkod_data_processor.create_metadata_csv(graph_db)
#nkod_data_processor.create_themes_csv(graph_db)
#nkod_data_processor.create_metadata_sql(sq_lite)
#nkod_data_processor.create_themes_sql(sq_lite)

## Indexing the keywords, titles and descriptions from the NKOD metadata (TODO: matching a dataset)

In [3]:
openai_embeddings = OpenAIEmbeddingProvider(model_name="text-embedding-3-large", dimensions=1536)
chroma_db = ChromaDb(nkod_data_processor.vectordb_path)

#nkod_data_processor.index_catalog_themes(sq_lite, google_embeddings, chroma_db)
#nkod_data_processor.index_catalog_metadata(sq_lite, google_embeddings, chroma_db, verbose=True)
print(chroma_db.list_collections())

['nkod_titles_cs', 'nkod_descriptions_cs', 'nkod_keywords_cs', 'nkod_descriptions_en', 'nkod_titles_en', 'nkod_keywords_en']


## Language detection, Timeframe detection and Query matching

In [4]:
model_name ="gpt-5"
openai_llm = OpenAILLMProvider(
    model_name=model_name,
    temperature=1.0
)
nkod_query_evaluator = NkodQueryMatcherEvaluator()
nkod_query_reranker = NkodQueryMatcherReranker()

k = 30
evaluation_dataset = "ofn_dataset_ofn_new.jsonl"
nkod_query_evaluator.evaluate_on_ofn_dataset(k, evaluation_dataset, chroma_db, nkod_data_processor, "cs", openai_embeddings, nkod_query_reranker, openai_llm)

Query 1/9
Desc: jednoduchý dotaz
Original query: Jaké jsou aktuality v Říčanech?
Cleaned query: jaké jsou aktuality v říčanech?
In titles: True -> 1/1 present | positions: [0]
In titles rerankedTrue -> 1/1 present | positions: [0]
Titles similarity matching: [(0.3168889284133911, 'aktuality města říčany'), (0.3986581563949585, 'aktuální stav na povodňových čidlech ústeckého kraje'), (0.41426151990890503, 'události ve městě říčany'), (0.42163896560668945, 'aktuality'), (0.42163896560668945, 'aktuality'), (0.42198145389556885, 'georizika - vodní toky'), (0.4572067856788635, 'vodní toky'), (0.46155452728271484, 'množství povrchových vod – údaje – recent'), (0.4615558385848999, 'množství povrchových vod – údaje – recent'), (0.470719575881958, 'pětisetletá voda q500 - záplavové území - liberecký kraj'), (0.47981637716293335, 'dvacetiletá voda q20 - záplavové území - liberecký kraj'), (0.48033618927001953, 'akumulace vod ve vodních nádržích 2021'), (0.48201149702072144, 'riparian zones - bře

In [12]:
import requests

url = "http://127.0.0.1:8000/match-query/"

payload = {
    "query": "události v říčanech",
    "llm_provider": "openai",
    "model_name": "gpt-5",
    "language": "cs",
    "embedding_provider": "openai"
}

response = requests.post(url, json=payload)

print("Status code:", response.status_code)
print("Response JSON:", response.json())

<Response [200]>
Status code: 200
Response JSON: {'text': 'konec'}
